# Boltz2 Confidence Ranking: 3k7u_H_X_A

Load the 20 `confidence_*.json` files for target `3k7u_H_X_A`, extract `confidence_score`, and rank models so higher confidence gets a better rank (`1` is best). Results are shown as a formatted table and a simple bar plot. The CSV text is displayed in the notebook and is not written to disk.

In [ ]:
from pathlib import Path
from html import escape
from io import StringIO
import csv
import json
import os
import re

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None
    display = None

TARGET = '3k7u_H_X_A'
NOTEBOOK_TMP = Path('/home/sujin/projects/cdr-scoring/cdr-code/notebooks/tmp')
(NOTEBOOK_TMP / '.cache').mkdir(parents=True, exist_ok=True)
(NOTEBOOK_TMP / '.matplotlib').mkdir(parents=True, exist_ok=True)
os.environ.setdefault('XDG_CACHE_HOME', str(NOTEBOOK_TMP / '.cache'))
os.environ.setdefault('MPLCONFIGDIR', str(NOTEBOOK_TMP / '.matplotlib'))

PRED_DIR = Path('/home/sujin/DB/h3-loop-modeling/ab_ag/1_tr_abag/11_boltz2/boltz_results_3k7u_H_X_A/predictions/3k7u_H_X_A')

confidence_files = sorted(
    PRED_DIR.glob('confidence*.json'),
    key=lambda p: int(re.search(r'_model_(\d+)\.json$', p.name).group(1)),
)

len(confidence_files), confidence_files[:3]


In [ ]:
rows = []

for path in confidence_files:
    model_match = re.search(r'_model_(\d+)\.json$', path.name)
    if model_match is None:
        raise ValueError(f'Could not parse model index from {path.name}')

    with path.open() as f:
        data = json.load(f)

    rows.append({
        'target': TARGET,
        'model': f"model_{int(model_match.group(1))}",
        'model_index': int(model_match.group(1)),
        'confidence_score': data['confidence_score'],
        'ptm': data.get('ptm'),
        'iptm': data.get('iptm'),
        'protein_iptm': data.get('protein_iptm'),
        'complex_plddt': data.get('complex_plddt'),
        'complex_iplddt': data.get('complex_iplddt'),
        'complex_pde': data.get('complex_pde'),
        'complex_ipde': data.get('complex_ipde'),
        'confidence_json': str(path),
    })

if len(rows) != 20:
    raise ValueError(f'Expected 20 confidence files, found {len(rows)}')

ranked_rows = sorted(rows, key=lambda row: (-row['confidence_score'], row['model_index']))
for rank, row in enumerate(ranked_rows, start=1):
    row['confidence_rank'] = rank

columns = [
    'target',
    'model',
    'model_index',
    'confidence_score',
    'confidence_rank',
    'ptm',
    'iptm',
    'protein_iptm',
    'complex_plddt',
    'complex_iplddt',
    'complex_pde',
    'complex_ipde',
    'confidence_json',
]

ranked_confidence = [{col: row[col] for col in columns} for row in ranked_rows]

summary = {
    'target': TARGET,
    'n_models': len(ranked_confidence),
    'best_model': ranked_confidence[0]['model'],
    'best_confidence_score': ranked_confidence[0]['confidence_score'],
    'worst_model': ranked_confidence[-1]['model'],
    'worst_confidence_score': ranked_confidence[-1]['confidence_score'],
}
summary


In [ ]:
display_columns = [
    'confidence_rank',
    'model',
    'confidence_score',
    'ptm',
    'iptm',
    'complex_plddt',
    'complex_iplddt',
    'complex_pde',
    'complex_ipde',
]

def fmt_value(value):
    if isinstance(value, float):
        return f'{value:.4f}'
    return escape(str(value))

score_min = min(row['confidence_score'] for row in ranked_confidence)
score_max = max(row['confidence_score'] for row in ranked_confidence)
score_span = score_max - score_min or 1.0

summary_html = f'''
<div style="font-family: system-ui, -apple-system, Segoe UI, sans-serif; margin: 0 0 14px 0;">
  <div style="font-size: 18px; font-weight: 700; margin-bottom: 6px;">{TARGET} confidence ranking</div>
  <div style="display: flex; gap: 10px; flex-wrap: wrap;">
    <div style="border: 1px solid #d7dde5; border-radius: 6px; padding: 8px 10px;"><b>models</b><br>{summary['n_models']}</div>
    <div style="border: 1px solid #d7dde5; border-radius: 6px; padding: 8px 10px;"><b>best</b><br>{summary['best_model']} ({summary['best_confidence_score']:.4f})</div>
    <div style="border: 1px solid #d7dde5; border-radius: 6px; padding: 8px 10px;"><b>worst</b><br>{summary['worst_model']} ({summary['worst_confidence_score']:.4f})</div>
  </div>
</div>
'''

header = ''.join(f'<th>{escape(col)}</th>' for col in display_columns)
body_rows = []
for row in ranked_confidence:
    cells = []
    for col in display_columns:
        value = row[col]
        if col == 'confidence_score':
            width = 12 + 88 * (value - score_min) / score_span
            cells.append(
                '<td>'
                f'<div style="font-variant-numeric: tabular-nums; margin-bottom: 3px;">{value:.4f}</div>'
                f'<div style="height: 7px; width: {width:.1f}%; background: #2f7f6f; border-radius: 3px;"></div>'
                '</td>'
            )
        elif col == 'confidence_rank':
            cells.append(f'<td style="font-weight: 700; text-align: center;">{value}</td>')
        else:
            cells.append(f'<td>{fmt_value(value)}</td>')
    body_rows.append('<tr>' + ''.join(cells) + '</tr>')

table_html = f'''
<style>
.confidence-table {{ border-collapse: collapse; font-family: system-ui, -apple-system, Segoe UI, sans-serif; font-size: 13px; width: 100%; }}
.confidence-table th {{ background: #eef2f6; color: #1f2933; border-bottom: 1px solid #cbd5df; padding: 7px 8px; text-align: left; position: sticky; top: 0; }}
.confidence-table td {{ border-bottom: 1px solid #e4e8ee; padding: 7px 8px; vertical-align: middle; }}
.confidence-table tr:nth-child(even) td {{ background: #fafbfc; }}
.confidence-table tr:hover td {{ background: #f0f6ff; }}
</style>
{summary_html}
<table class="confidence-table"><thead><tr>{header}</tr></thead><tbody>{''.join(body_rows)}</tbody></table>
'''

if display is not None and HTML is not None:
    display(HTML(table_html))
else:
    print(f"{TARGET} confidence ranking")
    print(f"best: {summary['best_model']} ({summary['best_confidence_score']:.4f})")
    print(f"worst: {summary['worst_model']} ({summary['worst_confidence_score']:.4f})")
    for row in ranked_confidence:
        print(f"#{row['confidence_rank']:>2} {row['model']:<8} {row['confidence_score']:.4f}")

# `ranked_confidence` remains available as a list of dictionaries for downstream code.


In [ ]:
import matplotlib.pyplot as plt

plot_rows = list(reversed(ranked_confidence))
models = [row['model'] for row in plot_rows]
scores = [row['confidence_score'] for row in plot_rows]
ranks = [row['confidence_rank'] for row in plot_rows]

fig_height = max(5, 0.32 * len(plot_rows))
fig, ax = plt.subplots(figsize=(8, fig_height), dpi=130)
colors = ['#2f7f6f' if rank <= 5 else '#8796a8' for rank in ranks]
ax.barh(models, scores, color=colors, height=0.68)

for y, score, rank in zip(models, scores, ranks):
    ax.text(score + 0.001, y, f'#{rank}  {score:.4f}', va='center', fontsize=9)

ax.set_title(f'{TARGET} Boltz2 confidence score ranking')
ax.set_xlabel('confidence_score')
ax.set_xlim(min(scores) - 0.005, max(scores) + 0.02)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', alpha=0.25)
plt.show()


In [ ]:
csv_buffer = StringIO()
writer = csv.DictWriter(csv_buffer, fieldnames=columns)
writer.writeheader()
writer.writerows(ranked_confidence)

confidence_csv = csv_buffer.getvalue()
print(confidence_csv)
